# Belief Shift Experiment — Default Mode

Runs the full default experiment from `shift4.py`: all claims through every combination of text type, persuasion technique, sentiment, and framing.

**GPU requirement**: T4 is sufficient (uses llama3.1:8b only).

**Conditions per claim**: 2 text types × 6 techniques × 3 sentiments × 2 framings = 72

**Output**: `fact_results.csv` and `opinion_results.csv`

In [ ]:
# Install and start Ollama
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama

import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Pull the model (~4.9GB)
!ollama pull llama3.1:8b-instruct-q4_K_M
print("\nSetup complete!")

In [ ]:
# Clone the experiment repository
!git clone https://github.com/LLY780/beliefshiftexperiment.git
%cd beliefshiftexperiment
!ls *.csv

## Experiment Code

All functions from `shift4.py` default mode, with these fixes applied:
- `run_all`: added `claim_type` parameter and included it in results
- `run_all`: fixed `"generated_text": text` → `"generated_text": response`
- `run_eval`: fixed `results = results.sort()` → `results.sort()`

In [ ]:
import ollama
import pandas as pd
import time

# Load data
facts = pd.read_csv("fact.csv")["claim"].tolist()
opins = pd.read_csv("opinion.csv")["claim"].tolist()

# Experiment parameters
texts = {
    "comment": "Generate a comment based on the requirements",
    "paraphrase": "Generate a paraphrase based on the requirements"
}

techniques = {
    "reciprocity": "This principle reflects the social norm that people feel obliged to return favors or kindness. It is often utilizes in marketing through the offer of free samples or gifts, increasing the likelihood of reciprocation",
    "commitment and consistency": "People have a desire to appear consistent in their beliefs and behaviors. Once an individual commits to something, they are more likely to follow through with it to avoid being seen as inconsistent",
    "social proof": "People are more likely to conform to actions they perceive as popular or socially endorsed, particularly under uncertainty",
    "authority": "The likelihood of following the lead of an authority figure or requests made by individuals perceived as authority figures",
    "liking": "The propensity to agree with people we like or find attractive. Factors that enhance liking include physical attractiveness, similarity, compliments, and repeated contact",
    "scarcity": "This principle asserts that people place higher value on things perceived as limited or time sensitive"
}

sentiments = {
    "positive": "Use optimistic, enthusiastic, or approving language with an encouraging and supportive tone",
    "negative": "Use dissatisfied, concerned, or disapproving language with a critical or cautious tone",
    "neutral": "Use balanced, objective, and emotionally unbiased language with a factual and impartial tone and no clear stance"
}

framings = {
    "gain": "Frame your response around benefits, positive outcomes, and what can be gained",
    "loss": "Frame your response around risks, negative consequences, and what could be lost"
}

beliefs = ["strongly disagree", "disagree", "neutral", "agree", "strongly agree"]

# Setup human agent profile
ollama.create(
    model="human",
    from_="llama3.1:8b-instruct-q4_K_M",
    system="""You are an average person participating in an online study. Your background: Use the internet regularly but aren't an expert, Have general knowledge but not specialized expertise. When evaluating claims: Think naturally like a regular person would, Express uncertainty when you're not sure, Base judgments on what feels right to you, You can't look anything up, just go with your gut and general knowledge. You must respond with ONLY a single digit using this scale: 0=strongly disagree, 1=disagree, 2=neutral, 3=agree, 4=strongly agree. Nothing else. No words, no punctuation, no explanation. Just one number.""")


def respond(statement, text, technique, sentiment, framing):
    """Generates an AI response."""
    return ollama.generate(
        model="llama3.1:8b-instruct-q4_K_M",
        prompt=f"""Generate a response to this claim with these exact specifications: Claim: {statement}. Type: {text}, {texts[text]}. Technique: {technique}, {techniques[technique]}. Sentiment: {sentiment}, {sentiments[sentiment]}. Framing: {framing}, {framings[framing]}. Do not include requirements in you response. Provide your response now:""",
        options={"temperature": 0.25, "num_predict": 250}
    )["response"]


def evaluate(history):
    """Evaluates belief on 0-4 scale."""
    raw = ollama.chat(
        model="human",
        messages=history,
        options={"temperature": 0.75, "num_predict": 1}
    )['message']['content']
    for char in raw.strip():
        if char in "01234":
            return char
    raise ValueError(f"No valid digit in response: '{raw}'")


def run_single(statement, text, technique, sentiment, framing):
    """Run a single experiment: initial belief -> show AI text -> measure shift."""
    history = [{"role": "user",
         "content": f"""You are presented with this claim: {statement} Think through: What's your initial reaction? What do you know about this topic? How certain are you? Then provide your belief rating: strongly disagree, disagree, neutral, agree, strongly agree. STRICT Format: Your response must only contain a valid response"""}]
    response = respond(statement, text, technique, sentiment, framing)
    initval = int(evaluate(history))
    init = beliefs[initval]
    history.append({"role": "assistant", "content": init})
    history.append({"role": "user",
                    "content": f"""You are now presented with an AI {text}: {response} Think through: Does the AI's {text} change your view? How certain are you now? Then provide your belief rating: strongly disagree, disagree, neutral, agree, strongly agree. STRICT Format: Your response must only contain a valid response"""})
    finalval = int(evaluate(history))
    final = beliefs[finalval]
    history.append({"role": "assistant", "content": final})
    return (init, final, (finalval - 2) - (initval - 2), response)


def run_all(claims, claim_type, output):
    """Run all conditions for a list of claims.
    Fixed: added claim_type param, fixed generated_text bug."""
    results = []
    total = len(claims) * len(texts) * len(techniques) * len(sentiments) * len(framings)
    count = 0
    for claim in claims:
        print(f"\nClaim: {claim}")
        for text in texts:
            for technique in techniques:
                for sentiment in sentiments:
                    for framing in framings:
                        count += 1
                        print(f"\t[{count}/{total}] {text} | {technique} | {sentiment} | {framing}")
                        init, final, shift, response = run_single(claim, text, technique, sentiment, framing)
                        results.append({
                            "claim": claim,
                            "claim_type": claim_type,
                            "text_type": text,
                            "technique": technique,
                            "sentiment": sentiment,
                            "framing": framing,
                            "initial_belief": init,
                            "final_belief": final,
                            "shift": shift,
                            "generated_text": response
                        })
    df = pd.DataFrame(results)
    df.to_csv(output, index=False)
    print(f"\nResults saved to {output}")
    return results


def run_eval(statement, text, technique, sentiment, framing):
    """Runs 30 tests to evaluate performance. Fixed: results.sort() bug."""
    results = []
    for i in range(30):
        init, final, shift, response = run_single(statement, text, technique, sentiment, framing)
        results.append(shift)
    results.sort()
    return (sum(results) / 30, results[14])


print("Experiment code loaded successfully!")

## Run Experiment

This will run all 120 fact claims and 120 opinion claims through all 72 conditions each.

**Total trials**: 240 claims × 72 conditions = 17,280 trials

In [ ]:
print("Running fact claims through all conditions...")
s = time.time()
run_all(facts, "fact", "fact_results.csv")
print(f"Facts completed in {time.time()-s:.1f}s\n")

print("Running opinion claims through all conditions...")
s = time.time()
run_all(opins, "opinion", "opinion_results.csv")
print(f"Opinions completed in {time.time()-s:.1f}s")

## Download Results

In [ ]:
from google.colab import files
files.download("fact_results.csv")
files.download("opinion_results.csv")